# Phase 17C Strategy Factory Transaction Visuals

This notebook is a static research and paper-watchlist report for Strategy Factory
transaction behaviour. It explains inferred target-weight changes for SPY, QQQ,
BTC-USD, GLD, TLT, and cash.

This is not live trading. It is not broker/API integration. It does not use real
money, does not optimise parameters, and does not promote any candidate.


## Load Data

The notebook reads existing Phase 17A/17B/17C reports plus the transaction
artefacts generated under `reports/strategy_factory/transactions/`.


In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd().resolve()
root = cwd if (cwd / "reports").exists() else cwd.parent
strategy_dir = root / "reports" / "strategy_factory"
transaction_dir = strategy_dir / "transactions"
chart_dir = transaction_dir / "charts"

files = {
    "phase17a_metrics": strategy_dir / "phase17a_strategy_factory_metrics.csv",
    "phase17b_friction": strategy_dir / "phase17b_friction_metrics.csv",
    "phase17b_btc_gap": strategy_dir / "phase17b_btc_weekend_gap_diagnostic.csv",
    "phase17c_watchlist": strategy_dir / "watchlist" / "phase17c_watchlist_candidates.csv",
    "transaction_ledger": transaction_dir / "strategy_transaction_ledger.csv",
    "drift_ledger": transaction_dir / "strategy_drift_rebalance_ledger.csv",
    "drift_summary": transaction_dir / "strategy_drift_rebalance_summary.csv",
    "trade_matrix": transaction_dir / "strategy_rebalance_trade_matrix.csv",
    "rebalance_summary": transaction_dir / "strategy_rebalance_summary.csv",
    "turnover_timeline": transaction_dir / "strategy_turnover_timeline.csv",
    "allocation_long": transaction_dir / "strategy_asset_allocation_long.csv",
    "latest_allocations": transaction_dir / "strategy_latest_allocations.csv",
}
pd.DataFrame(
    [{"name": name, "path": str(path), "exists": path.exists()} for name, path in files.items()]
)


In [ ]:
metrics = pd.read_csv(files["phase17a_metrics"])
friction = pd.read_csv(files["phase17b_friction"])
btc_gap = pd.read_csv(files["phase17b_btc_gap"])
watchlist = pd.read_csv(files["phase17c_watchlist"])
ledger = pd.read_csv(files["transaction_ledger"])
drift_ledger = pd.read_csv(files["drift_ledger"])
drift_summary = pd.read_csv(files["drift_summary"])
trade_matrix = pd.read_csv(files["trade_matrix"])
summary = pd.read_csv(files["rebalance_summary"])
turnover = pd.read_csv(files["turnover_timeline"])
allocation = pd.read_csv(files["allocation_long"])
latest_allocations = pd.read_csv(files["latest_allocations"])

watchlist_ids = watchlist["candidate_id"].astype(str).tolist()
all_strategy_ids = sorted(allocation["strategy_id"].astype(str).unique())
watchlist_ids


## Strategy Explanation

- `sf_spy_buy_hold`: holds 100% SPY.
- `sf_spy_qqq_60_40_monthly_rebalanced`: targets 60% SPY and 40% QQQ with monthly rebalancing.
- `sf_spy_qqq_tactical_momentum`: chooses SPY or QQQ using a fixed trailing momentum rule, with cash when both are negative.
- `sf_spy_qqq_gld_tlt_risk_off_rotation`: rotates between SPY/QQQ risk-on exposure and GLD/TLT/cash risk-off exposure.
- `sf_spy_core_phase6_overlay_satellite_qqq`: links a SPY overlay-style core to a capped QQQ satellite when risk-on.
- `sf_spy_qqq_btc_capped_offensive`: holds an SPY/QQQ base with a capped BTC sleeve when BTC momentum and SPY risk-on conditions are positive.


## Performance Overview

These tables and charts describe research performance only. They are not
promotion decisions and do not alter the current operational paper baseline.


In [ ]:
overview_cols = [
    "strategy",
    "end_value",
    "total_return_pct",
    "cagr_pct",
    "volatility_pct",
    "max_drawdown_pct",
    "calmar",
]
metrics[overview_cols].sort_values("end_value", ascending=False)


In [ ]:
plot_frame = metrics.set_index("strategy")[["end_value", "cagr_pct", "max_drawdown_pct", "calmar"]]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, column in zip(axes.ravel(), plot_frame.columns):
    plot_frame[column].sort_values().plot(kind="barh", ax=ax)
    ax.set_title(column)
    ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout()
plt.show()


## Allocation Timelines

The Strategy Factory transaction ledger is inferred from target allocation changes.
These are not broker fills.


In [ ]:
display(Image(filename=str(chart_dir / "allocation_timeline_by_strategy.png")))
display(Image(filename=str(chart_dir / "latest_allocation_snapshot.png")))


In [ ]:
allocation.loc[
    allocation["strategy_id"].isin(watchlist_ids),
    ["date", "strategy_id", "asset", "weight"],
].tail(30)


## Target-weight changes vs actual rebalance trades

The target-weight ledger shows signal or allocation changes. For example, the BTC
capped strategy has explicit BTC entries and exits when the target BTC sleeve moves
between 0% and 10%.

The drift-rebalance ledger estimates implementation trades. It lets current weights
drift with asset returns between rebalance dates, then calculates the trades needed to
restore the strategy target. This makes constant-target strategies like SPY/QQQ 60/40
visible: even if the target stays 60/40, market returns push the actual portfolio away
from 60/40 and monthly rebalancing creates buy/sell trades.


In [ ]:
target_change_rows = ledger.groupby("strategy_id").size().rename("target_change_rows")
drift_rows = drift_ledger.groupby("strategy_id").size().rename("drift_rebalance_rows")
pd.concat([target_change_rows, drift_rows], axis=1).fillna(0).astype(int)


## Transaction Ledger

The ledger shows inferred BUY_ENTRY, BUY_INCREASE, SELL_REDUCE, SELL_EXIT, and
CASH_ALLOCATION_CHANGE rows. Notional changes use a $10,000 interpretation baseline.


In [ ]:
ledger.head(50)


In [ ]:
ledger.tail(50)


In [ ]:
by_strategy = ledger.groupby("strategy_id").size().rename("transaction_rows")
by_asset = ledger.groupby("asset").size().rename("transaction_rows")
buy_sell = ledger.groupby(["strategy_id", "transaction_direction"]).size().unstack(fill_value=0)
by_strategy, by_asset, buy_sell


In [ ]:
display(Image(filename=str(chart_dir / "transaction_count_by_asset.png")))


## Drift-based Rebalance Ledger

These rows estimate the trades needed to restore target weights after asset returns
cause holdings to drift. They are implementation estimates, not fills.


In [ ]:
drift_ledger.head(50)


In [ ]:
drift_ledger.tail(50)


In [ ]:
drift_summary.sort_values("rebalance_trade_rows", ascending=False)


In [ ]:
display(Image(filename=str(chart_dir / "drift_rebalance_trades_by_strategy.png")))
display(Image(filename=str(chart_dir / "drift_rebalance_trades_by_asset.png")))
display(Image(filename=str(chart_dir / "drift_rebalance_notional_by_asset.png")))


## SPY/QQQ 60/40 Rebalance Inspection

This section shows the monthly implementation trades for the constant-target 60/40
candidate. Positive notional buys an asset back to target; negative notional sells an
overweight asset back to target.


In [ ]:
strategy_6040 = "sf_spy_qqq_60_40_monthly_rebalanced"
trades_6040 = drift_ledger[
    (drift_ledger["strategy_id"] == strategy_6040)
    & (drift_ledger["asset"].isin(["SPY", "QQQ"]))
].copy()
display(Image(filename=str(chart_dir / "spy_qqq_60_40_rebalance_timeline.png")))
trades_6040.tail(30)


## BTC Capped Rebalance Inspection

BTC entries and exits appear in the target-change ledger. Regular rebalance trades while
BTC is active appear in the drift ledger.


In [ ]:
btc_target_changes = ledger[
    (ledger["strategy_id"] == "sf_spy_qqq_btc_capped_offensive")
    & (ledger["asset"] == "BTC-USD")
].copy()
btc_drift_trades = drift_ledger[
    (drift_ledger["strategy_id"] == "sf_spy_qqq_btc_capped_offensive")
    & (drift_ledger["asset"] == "BTC-USD")
].copy()
btc_target_changes.tail(20), btc_drift_trades.tail(20)


## Trade Burden Comparison

This compares which strategies and assets create the most estimated implementation
activity.


In [ ]:
trade_matrix.sort_values("total_abs_trade_notional_per_10000", ascending=False).head(25)


## Turnover Visualisation

Turnover is the sum of absolute target-weight changes by strategy and rebalance date.
Large spikes identify periods where the candidate changed exposure materially.


In [ ]:
display(Image(filename=str(chart_dir / "turnover_timeline_by_strategy.png")))
high_turnover = turnover.sort_values("turnover", ascending=False).head(25)
high_turnover


## BTC-Specific Inspection

BTC strategy returns are evaluated on ETF-common trading dates in the Strategy Factory
frame. BTC weekend observations and weekend gap risk are diagnostic context and are not
directly represented in ETF-common-date strategy returns.


In [ ]:
display(Image(filename=str(chart_dir / "btc_weight_timeline.png")))
btc_gap


## Watchlist Interpretation

- Clean growth watchlist: SPY/QQQ 60/40.
- Baseline-linked growth watchlist: Phase6-style SPY overlay plus QQQ satellite.
- High-growth/high-caveat watchlist: SPY/QQQ/BTC capped offensive.


In [ ]:
watchlist[
    [
        "candidate_id",
        "watchlist_role",
        "low_friction_cagr_pct",
        "realistic_stress_cagr_pct",
        "max_drawdown_pct",
        "rolling_3y_candidate_beats_spy_pct",
        "btc_cap_dependency_flag",
        "promotion_allowed",
        "paper_watchlist_only",
    ]
]


## Current Paper Relevance

Phase 16 paper preview currently represents only the Phase6 SPY overlay baseline.
The 60/40 and BTC Strategy Factory candidates are research watchlist candidates and
are not represented in active paper order preview files.


## Conclusion

Target-weight changes are signal/allocation changes. Drift-based rebalance trades are
estimated implementation trades. Neither are broker fills.

No candidate is promoted. There is no live trading, no real money, and no broker/API
execution. The next operational step is watchlist paper-preview integration or recurring
signal hardening, not live deployment.
